# The Snowflake Model Registry

### Snowpark ML Operations allows you to manage models regardless or origin!

The model registry stores machine learning models as first-class schema-level objects in Snowflake so they can easily be found and used by others in your organization. You can create registries, and store models in them, using classes in the Snowpark ML library. 

Models can have multiple versions, and you can designate a version as the default.

<img src="../../images/Snowpark_ML_Operations.png" alt="MLAPIQuery" style="width:65%;display:block;margin-left:10%;" />

<a id="topics"></a>
### Topics in this Lesson


1. [Model deployment using Model Registry](#using_Model_Registry)
   1.  [Let's load and apply the ML pipeline we built previously](#load_and_apply)   
   1. [Opening a model registry](#opening)  
    1. [Logging a model ](#Logging)  

<a id="using_Model_Registry"></a>
### 1. Model deployment using Model Registry

To start, update the snowpark_ml_python package, in case needed.

In [ ]:
pip install snowflake-ml-python -U

Now, with Snowpark ML's model registry, we have a Snowflake native model versioning and deployment framework. This allows us to log models, tag parameters and metrics, track metadata, create versions, and ultimately use our models in a Snowflake warehouse or Snowpark Container Service for batch scoring tasks. All of this can even be achieved inside a Snowflake Notebook without the data ever leaving Snowflake. 

The Snowpark Model Registry supports the following types of models.
- [Snowpark ML Modeling](https://docs.snowflake.com/en/developer-guide/snowpark-ml/snowpark-ml-modeling)
- scikit-learn
- XGBoost
- LightGBM
- CatBoost
- PyTorch
- TensorFlow
- MLFlow PyFunc
- Sentence Transformer
- Hugging Face pipeline
- Other types of models via the snowflake.ml.model.CustomModel class

In [2]:
# Snowpark for Python
from snowflake.snowpark import Session
from snowflake.snowpark.version import VERSION
from snowflake.snowpark.functions import udf
import snowflake.snowpark.functions as F

# Snowpark ML
import snowflake.ml.modeling.preprocessing as SNOWML
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.xgboost import XGBRegressor
from snowflake.ml.modeling.model_selection import GridSearchCV
from snowflake.ml.registry import Registry
from snowflake.ml._internal.utils import identifier

# data science libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.ml.modeling.metrics import mean_absolute_percentage_error

# misc
import json
import joblib
import cachetools

# warning suppresion
import warnings; warnings.simplefilter('ignore')


/Users/richardkirk/SourceCode/snowflake-datasciencelabs-jupyter/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We need a version of 1.5 of higher of `snowflake-ml-python`. 

If you are running an older version run:
`pip install snowflake-ml-python --upgrade`

MyNote: My version of snowflake-ml-python is 1.40.0, this is actually higher than 1.5, which apparently was an old versionining format.

#### Connect and create a `Session`

The following cell connects to your Snowflake account and creates an instance of `Session`. 

*You needn't modify anything in this cell. Just run it.*
> &#10071; **Success requires that you have already completed the key pair authentication notebook in UTILS.**

In [3]:
# Run utils notebook
%run ../../utils/ds_utils_python_MINE.ipynb

# Connect to Snowflake and create a Session object named session
# session = create_session()

In [4]:
# My code for Snowflake account connection

CONFIG_DIR = '/Users/richardkirk/.ssh'
CONFIGFILE = CONFIG_DIR + '/sf_config'


# Load configuration file
with open(CONFIGFILE) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

In [5]:
# Hard code the lesson name
lesson_name = "SNOWFLAKE_ML_PY"

# Create the context items for this lesson
# We add an extra parameter here to avoid to recreate the schema where our data resides
lesson = confirm_or_create_lesson_context(session, lesson_name, True)

Creation of context items could take a few moments... be patient
The current user is: RKIRK
Query tag is: Data Science: RKIRK - SNOWFLAKE_ML_PY
-------------------------------------------------
|"Created warehouse RKIRK_WH"                   |
-------------------------------------------------
|RKIRK_WH already exists, statement succeeded.  |
-------------------------------------------------

------------------------------------
|"Altered warehouse RKIRK_WH"      |
------------------------------------
|Statement executed successfully.  |
------------------------------------

Setting current warehouse to RKIRK_WH
Creating database RKIRK_DB
-------------------------------------------------
|"Created database RKIRK_DB"                    |
-------------------------------------------------
|RKIRK_DB already exists, statement succeeded.  |
-------------------------------------------------

---------------------------------------------------------------
|"Created schema RKIRK_DB.SNOWFLAKE_ML_

In [6]:
snowflake_environment = session.sql('SELECT current_user(), current_version()').collect()
snowpark_version = VERSION
current_schema = session.get_current_schema()
current_database = session.get_current_database()
# Current Environment Details
print('\nConnection Established with the following parameters:')
print('User                        : {}'.format(snowflake_environment[0][0]))
print('Role                        : {}'.format(session.get_current_role()))
print('Database                    : {}'.format(session.get_current_database()))
print('Schema                      : {}'.format(session.get_current_schema()))
print('Warehouse                   : {}'.format(session.get_current_warehouse()))
print('Snowflake version           : {}'.format(snowflake_environment[0][1]))
print('Snowpark for Python version : {}.{}.{}'.format(snowpark_version[0],snowpark_version[1],snowpark_version[2]))


Connection Established with the following parameters:
User                        : RKIRK
Role                        : "ACCOUNTADMIN"
Database                    : "RKIRK_DB"
Schema                      : "SNOWFLAKE_ML_PY_LESSON"
Warehouse                   : "RKIRK_WH"
Snowflake version           : 10.19.100
Snowpark for Python version : 1.50.1


<a id="load_and_apply"></a>
## 1.1 Let's load the data

In [7]:
# Load in the data
diabetes_df = session.table(f"{current_database}.{current_schema}.diabetes")
diabetes_df.show()

-------------------------------------------------------------------------------------------------------------------------------------------------------
|"AGE"  |"SEX"   |"BMI"       |"BP"  |"BLOOD_SERUM1"  |"BLOOD_SERUM2"  |"BLOOD_SERUM3"  |"BLOOD_SERUM4"  |"BLOOD_SERUM5"  |"BLOOD_SERUM6"  |"TARGET"  |
-------------------------------------------------------------------------------------------------------------------------------------------------------
|59     |Female  |obese       |101   |157             |93              |38              |4               |49              |87              |151       |
|48     |Male    |normal      |87    |183             |103             |70              |3               |39              |69              |75        |
|72     |Female  |obese       |93    |156             |94              |41              |4               |47              |85              |141       |
|24     |Male    |overweight  |84    |198             |131             |40              

We are going to build an ML pipeline and store it in the Snowpark Model Registry for retrieval and use. As of this writing, the registry will not host a pipeline that includes a custom class. 

So, for this example, we'll alter the data in the diabates dataset so that the SEX column contains only uppercase labels. This allows the one-hot encoder to use those values as parts of new feature names, without mixed-case naming issues. We're doing this in advance, because the ML pipeline going in the Model Registry cannot have the our custom StringToUpper custom transformer.

**MyNote: Apparently this restriction no longer applies** - see my Obsidian note - snowflake-ML/Model Registry:  "Custom Class in Pipeline in model registry"

In [8]:
# Convert column to uppercase
original_column_order = diabetes_df.columns
diabetes_upper_df = (diabetes_df.
               withColumn('SEX', F.upper(diabetes_df['SEX'])).
               select(original_column_order)
              )

# Save the converted data to a new table
diabetes_upper_df.write.mode('overwrite').save_as_table(f"{current_database}.{current_schema}.diabetes_upper")

In [9]:
diabetes_upper_df.show(5)

-------------------------------------------------------------------------------------------------------------------------------------------------------
|"AGE"  |"SEX"   |"BMI"       |"BP"  |"BLOOD_SERUM1"  |"BLOOD_SERUM2"  |"BLOOD_SERUM3"  |"BLOOD_SERUM4"  |"BLOOD_SERUM5"  |"BLOOD_SERUM6"  |"TARGET"  |
-------------------------------------------------------------------------------------------------------------------------------------------------------
|59     |FEMALE  |obese       |101   |157             |93              |38              |4               |49              |87              |151       |
|48     |MALE    |normal      |87    |183             |103             |70              |3               |39              |69              |75        |
|72     |FEMALE  |obese       |93    |156             |94              |41              |4               |47              |85              |141       |
|24     |MALE    |overweight  |84    |198             |131             |40              

### Define an ML pipeline to use the new dataset

This pipeline will not include the custom StringToUpper class. 

In [10]:
# Categorize features for preprocessing

# Numerical columns to undergo MinMax scaling
MINMAX_COLUMNS = ["AGE","BLOOD_SERUM1", "BLOOD_SERUM2", "BLOOD_SERUM3", "BLOOD_SERUM4", "BLOOD_SERUM5", "BLOOD_SERUM6"]

# Columns for one-hot encoding
CATEGORICAL_COLUMNS = ["SEX"]

# Columns for ordinal encoding, along with order
ORDINAL_COLUMNS = ["BMI"]
ORDINAL_CATEGORIES = {
    "BMI": np.array(["underweight", "normal", "overweight", "obese", "extremely_obese"])

}

# Define the full ML pipeline, including the grid search.
ml_pipeline = Pipeline(
    steps=[
            (
                "MMS",
                SNOWML.MinMaxScaler(
                    clip=True,
                    input_cols=MINMAX_COLUMNS,
                    output_cols=MINMAX_COLUMNS,    # Replace original columns
                )
            ),
            (
                "OE",
                SNOWML.OrdinalEncoder(
                    input_cols=ORDINAL_COLUMNS,
                    output_cols=ORDINAL_COLUMNS,     # Replace original columns
                    categories=ORDINAL_CATEGORIES,   # Ordinal information
                )
            ),
            (
                "OHE",
                SNOWML.OneHotEncoder(
                    input_cols=CATEGORICAL_COLUMNS,
                    output_cols=CATEGORICAL_COLUMNS,
                    drop_input_cols=True              # Remove original column
                )
            ),
            (
                "XGBOOST",
                XGBRegressor(
                    label_cols='TARGET',
                    output_cols='PREDICTION',
                    n_estimators=50,                  # Learned from grid search previously
                    learning_rate=0.05                # Learned from grid search previously
                )
            )
    ]
)

#### Split the data into a test and train df

In [11]:
# Split the data into train and test sets
diabetes_upper_train_df, diabetes_upper_test_df = diabetes_upper_df.random_split(weights=[0.9, 0.1], seed=0)

In [11]:
# Train the ML pipeline using the training data as before
ml_pipeline = ml_pipeline.fit(diabetes_upper_train_df)

Package 'snowflake-telemetry-python' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.
The version of package 'numpy' in the local environment is 2.4.6, which does not fit the criteria for the requirement 'numpy==1.26.4'. Your UDF might not work when the package version is different between the server and your local environment.


In [12]:
# Apply the pipeline to perform inferencing test data
test_result_df = ml_pipeline.predict(diabetes_upper_test_df)

The version of package 'numpy' in the local environment is 2.4.6, which does not fit the criteria for the requirement 'numpy==1.26.4'. Your UDF might not work when the package version is different between the server and your local environment.


In [13]:
test_result_df.select('target', 'prediction').show(5)

---------------------------------
|"TARGET"  |"PREDICTION"        |
---------------------------------
|168       |91.17017364501953   |
|245       |221.91995239257812  |
|137       |100.66577911376953  |
|131       |113.38017272949219  |
|202       |192.20379638671875  |
---------------------------------



Evaluate this model once again, as before

In [14]:
mape_from_test_eval = mean_absolute_percentage_error(df=test_result_df,
                                                     y_true_col_names='TARGET',
                                                     y_pred_col_names='PREDICTION')
print(f"Mean absolute percentage error: {mape_from_test_eval}")

Mean absolute percentage error: 0.36113426930236914


<a id="opening"></a>
## 1.2 Opening a model registry
Before you can create or modify models in the [model registry](https://docs.snowflake.com/developer-guide/snowpark-ml/reference/latest/api/registry/snowflake.ml.registry.Registry#snowflake.ml.registry.Registry), you must open the registry. Opening the registry returns a reference to it, which you can then use to add new models and obtain references to existing models.

Models are first-class Snowflake objects and can be organized within a database and schema along with other Snowflake objects. The Snowflake Model Registry provides a Python class for managing models within a schema. Thus, any Snowflake schema can be used as a registry.



In [12]:
# We are opening a Model Registry in our current schema
session.get_current_schema()

'"SNOWFLAKE_ML_PY_LESSON"'

In [13]:
db = identifier._get_unescaped_name(session.get_current_database())
schema = identifier._get_unescaped_name(session.get_current_schema())

# Open a registry, we now have a reference to it.
registry = Registry(session=session, database_name=db, schema_name=schema)

# Get sample input data to pass into the registry logging function
X = diabetes_upper_df.drop('target').limit(100)

<a id="Logging"></a>
## 1.3 Logging a model 

Adding a model to the registry is called logging the model. Log a model by calling the registry’s log_model method. 

This method:

- Serializes the model, a Python object, and creates a Snowflake model object from it.

- Adds metadata, such as a description, to the model as specified in the log_model call.

Next, we will log our model.
The function `snowflake.ml.Registry.log_model(...)` takes the following aguments:
- **model** - Model object of supported types such as Scikit-learn, XGBoost, Snowpark ML, PyTorch, TorchScript, Tensorflow, Tensorflow Keras, MLFlow, HuggingFace Pipeline, or Custom Model.
- **model_name** - Name to identify the model.
- **version_name** - Version identifier for the model. Combination of model_name and version_name must be unique.
- **comment** - Comment associated with the model version. Defaults to None.
- **metrics** - A JSON serializable dictionary containing metrics linked to the model version. Defaults to None.
- **signatures** - Model data signatures for inputs and outputs for various target methods. Defaults to None.
- **sample_input_data** - Sample input data to infer model signatures from. Defaults to None.
- **conda_dependencies** - List of Conda package specifications. Defaults to None.
- **pip_requirements** - List of Pip package specifications. Defaults to None.
- **python_version** - Python version in which the model is run. Defaults to None.
- **code_paths** - List of directories containing code to import. Defaults to None.
- **ext_modules** List of external modules to pickle with the model object. Only supported when logging the following types of model: Scikit-learn, Snowpark ML, PyTorch, TorchScript and Custom Model. Defaults to None.
- **options** - Optional. Additional model saving options.

In [14]:
# Define model name
MODEL_NAME = "DIABETES_MODEL"

# (Delete model if present from a previous notebook run.)
mlist = registry.show_models()
if len(mlist) > 0:                                            # Any models at all
    if mlist[mlist['name']==MODEL_NAME]["name"].count() > 0:  # This model
        registry.delete_model(MODEL_NAME)

In [18]:
# Register V1 of the model
registry.log_model(
    model_name=MODEL_NAME,
    version_name='V1',
    model=ml_pipeline,
    sample_input_data=X
)

Model logged successfully.: 100%|██████████| 6/6 [01:11<00:00, 11.88s/it]                          


ModelVersion(
  name='DIABETES_MODEL',
  version='V1',
)

`log_model` returns a `snowflake.ml.model.ModelVersion` object, which represents the version of the model that was added to the registry.

Each model may have any number of versions. Currently it has a maximum of 50 versions. 

When logging the model, only the model_name and the model are required. The others are optional. 

For production use, let's train the model again with *all* available data--not just the test data.

In [15]:
ml_pipeline_v2 = ml_pipeline.fit(diabetes_upper_df)

Package 'snowflake-telemetry-python' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.
The version of package 'numpy' in the local environment is 2.4.6, which does not fit the criteria for the requirement 'numpy==2.4.4'. Your UDF might not work when the package version is different between the server and your local environment.


In [16]:
# Log the model
registry.log_model(
    model_name=MODEL_NAME,
    version_name='V2',
    model=ml_pipeline_v2,
    sample_input_data=X
)

Model logged successfully.: 100%|██████████| 6/6 [00:47<00:00,  7.86s/it]                          


ModelVersion(
  name='DIABETES_MODEL',
  version='V2',
)

In [17]:
# Let's log it again, but with a different version number.
registry.log_model(
    model_name=MODEL_NAME,
    version_name='V3',
    model=ml_pipeline_v2,
    sample_input_data=X
)

Model logged successfully.: 100%|██████████| 6/6 [00:24<00:00,  4.07s/it]                          


ModelVersion(
  name='DIABETES_MODEL',
  version='V3',
)

After a model has been logged, its artifacts (the files backing the model, including its serialized Python objects and various metadata files such as its manifest) are available on an internal stage.

This is not a conventional stage visible when listing all the stages, neither can you query it like a normal stage. 

In [18]:
# A version inside a model can be specified by a URL of the form snow://model/<model_name>/versions/<version_name>/.
session.sql("LIST 'snow://model/diabetes_model/versions/V3/'").show(25, 80)

----------------------------------------------------------------------------------------------------------------------------------------
|"name"                                                    |"size"  |"md5"                             |"last_modified"                |
----------------------------------------------------------------------------------------------------------------------------------------
|versions/V3/MANIFEST.yml                                  |1091    |5f56b275b66f76ec6be19eafb056c2d0  |Wed, 27 May 2026 16:44:07 GMT  |
|versions/V3/functions/predict.py                          |3096    |3c091c1a8b7d2f4867e4ca2966bc20b6  |Wed, 27 May 2026 16:44:07 GMT  |
|versions/V3/model/env/conda.yml                           |248     |e43bbb3bc037d8333cc649c3e50e1a49  |Wed, 27 May 2026 16:44:07 GMT  |
|versions/V3/model/env/requirements.txt                    |0       |d41d8cd98f00b204e9800998ecf8427e  |Wed, 27 May 2026 16:44:08 GMT  |
|versions/V3/model/model.yaml            

###MyNote: Can also run this as SQL direct in snowsight:
```sql
use schema rkirk_DB.SNOWFLAKE_ML_PY_LESSON;
LIST 'snow://model/diabetes_model/versions/V3/';
LIST 'snow://model/diabetes_model/versions/V2/';
```

You can use GET to retrieve the contents of a specific file in the stage. Alternatively, you can get the collection of files for a model version through `ModelVersion.export` as we will see further down in this notebook.

In [19]:
session.file.get('snow://model/diabetes_model/versions/V3/MANIFEST.yml', 'model_artifacts')

[GetResult(file='34564_145875744/MANIFEST.yml', size=1091, status='DOWNLOADED', message='')]

#### Listing models

Below we list the currently logged models. 

Notice that this is a Pandas dataframe. 

In [19]:
registry.show_models()

,created_on,name,model_type,database_name,schema_name,comment,owner,default_version_name,versions,aliases
0,2026-05-27 09:28:42.427000-07:00,DIABETES_MODEL,USER_MODEL,RKIRK_DB,SNOWFLAKE_ML_PY_LESSON,None,ACCOUNTADMIN,V2,"[""V2"",""V3""]","{""DEFAULT"":""V2"",""FIRST"":""V2"",""LAST"":""V3""}"


###MyNote: Or as SQL direct in snowsight:
```sql
show models in schema rkirk_DB.SNOWFLAKE_ML_PY_LESSON;
```


In [20]:
# This will only return a simple list of models
registry.models()

In [20]:
# This will return a specific model
registry.get_model("DIABETES_MODEL")

We can see what the default model is when we have multiple versions with the same model name:

In [21]:
registry.get_model(MODEL_NAME).default.version_name

'V2'

In [ ]:
mod = registry.get_model("DIABETES_MODEL")

**&#128221; Note:** You might expect deployment of the model at some point in time. However, it is not necessary to explicitly deploy a model in Snowflake. 

Check the [documentation](https://docs.snowflake.com/en/developer-guide/snowpark-ml/snowpark-ml-mlops-model-registry-api-diff#deploying-a-model)

We've logged the model in the Registry. Now using the model for inference is easy.

To use the model we need to select the specific model.

The method `.get_model()` will return a `Model` object. It will not specify what version we are going to use.

The `Model` object will have the following methods: 
- **delete_version()**
- **get_tag()**
- **rename()**
- **set_tag()**
- **show_tags()**

- **show_versions()**
- **unset_tag()**
- **version()**
- **versions()**

Notice that there is no `.run()` method. That is because we still need to select the version.



In [22]:
# Selecting the model we are going to use
model_version2 = registry.get_model(MODEL_NAME).version('V2')

print(f"The class of model_version2 is {type(model_version2)}")

The class of model_version2 is <class 'snowflake.ml.model._client.model.model_version_impl.ModelVersion'>


We now have an instance of a `ModelVersion` class. This has the following methods:
- **delete_metric()** - Delete a metric from metric storage.
- **export()** - Export model files to a local directory.
- **get_metric()** - Get the value of a specific metric.
- **load()** - Load the underlying original Python object back from a model.
- **set_metric()** - Set the value of a specific metric.
- **show_functions()** - Show all functions information in a model version that is callable.
- **show_metrics()** - Show all metrics logged with the model version.

and

- **run()**


Now that we have a `ModelVersion`. Let's export the files we saw earlier in this notebook.

In [ ]:
import os, shutil
export_path = os.getcwd() + '/model_export'

# Clean up /model_export folder if it exists from a previous run
if os.path.isdir(export_path):
    shutil.rmtree(export_path)
    
# Create empty export folder
os.mkdir(export_path)

# Export the model to the local folder
model_version2.export(export_path)

# Now check what has been exported into that folder!

#MyNote: exports here: /Users/richardkirk/SourceCode/snowflake-datasciencelabs-jupyter/7_train_models/Lectures/model_export

The function `snowflake.ml.model.ModelVersion.run(...)` takes the following aguments:
- **X** -  The input data, which could be a pandas DataFrame or Snowpark DataFrame.
- **function_name** – The function name to run. It is the name used to call a function in SQL. Defaults to None. It can only be None if there is only 1 method.
- **partition_column** - The partition column name to partition by.
- **strict_input_validation** - Enable stricter validation for the input data. This will result value range based type validation to make sure your input data won’t overflow when providing to the model.


In [24]:
# .run 
result_sdf2 = model_version2.run(diabetes_upper_test_df, function_name="predict")

In [25]:
result_sdf2.select('target', 'prediction').show()

---------------------------------
|"TARGET"  |"PREDICTION"        |
---------------------------------
|168       |114.51654052734375  |
|245       |239.692626953125    |
|137       |118.56708526611328  |
|131       |126.08529663085938  |
|202       |194.35316467285156  |
|42        |72.1611099243164    |
|48        |91.8325424194336    |
|96        |95.78530883789062   |
|279       |229.85279846191406  |
|173       |187.99615478515625  |
---------------------------------



We can also call these functions directly in SQL instead. 

First we quickly create a temporary table to store the test data set. 

In [29]:
# diabetes_upper_test_df.write.mode('overwrite').save_as_table('diabetes_upper_test', table_type="temporary")

diabetes_upper_test_df.write.mode('overwrite').save_as_table('diabetes_upper_test')


Now we can use our model in SQL and score the temporary table DIABETES_TEST:

In [27]:
# for the default version:
session.sql(f"SELECT TARGET, DIABETES_MODEL!predict(* EXCLUDE target)['PREDICTION']::INT AS prediction FROM diabetes_upper_test").show(5)

---------------------------
|"TARGET"  |"PREDICTION"  |
---------------------------
|168       |115           |
|245       |240           |
|137       |119           |
|131       |126           |
|202       |194           |
---------------------------



###MyNote: Running direct as SQL in snowsight
```sql
--nb have to ensure create diabetes_upper_test as regular table (not as temp table)
use schema rkirk_DB.SNOWFLAKE_ML_PY_LESSON;
SELECT TARGET, DIABETES_MODEL!predict(* EXCLUDE target)['PREDICTION']::INT AS prediction FROM snowflake_ml_py_lesson.diabetes_upper_test;
```



In [28]:
# for any other version (for example V2 below):
session.sql(f"WITH model_version_alias AS MODEL {MODEL_NAME} VERSION V2 SELECT target, model_version_alias!predict(* EXCLUDE target)['PREDICTION']::INT as prediction from diabetes_upper_test").show(5)

---------------------------
|"TARGET"  |"PREDICTION"  |
---------------------------
|168       |115           |
|245       |240           |
|137       |119           |
|131       |126           |
|202       |194           |
---------------------------



### Cleanup

Delete the model from the registry, in case you want to run this notebook again.
(To keep the model, comment out the following line of code.)

In [30]:
registry.delete_model('DIABETES_MODEL')

In [ ]:
# close_session_and_clean_up(get_lesson())